# Nashik Simhastha — Amrit Snan site readiness

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nnk-coder/nashik-kumbh-readiness/blob/main/notebooks/nashik_readiness.ipynb)

Measures the open ground at each bathing site from OpenStreetMap, works out
what it can process in the busiest hour of an Amrit Snan, says where the
people it cannot process are supposed to go, and checks toilets and drinking
water against CPHEEO's norms.

**Runtime:** about 5 minutes. **Needs:** nothing but an internet connection —
no account, no API key, no credentials.

---


## 1. Setup


In [ ]:
REPO_URL = 'https://github.com/nnk-coder/nashik-kumbh-readiness.git'

import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('nashik-kumbh-readiness'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)
if os.path.exists('nashik-kumbh-readiness'):
    os.chdir('nashik-kumbh-readiness')
sys.path.insert(0, os.getcwd())
print('working in', os.getcwd())


In [ ]:
!pip -q install shapely requests pandas pytest


## 2. Verify the rubric before spending any effort on data

The whole analysis layer is pure Python with no network in it, so it tests
in under a second. If the arithmetic is wrong, better data will not save it.


In [ ]:
!python -m pytest -q


## 3. Pull the map

One Overpass query per site: buildings, water, roads, open space, toilets and
drinking water within 1.5 km. Overpass is a free public service — be polite
with it and expect the odd retry.


In [ ]:
from src import assess, card, site, config as C
import datetime as dt

for s in C.SITES:
    print(f"{s['name']:<22} {s['city']:<16} weight {s['weight']:.0%}")


In [ ]:
records = assess.assess_all()
paths = assess.save_results(records)
print(paths)


## 4. Read the result


In [ ]:
import pandas as pd
df = pd.read_csv('outputs/results.csv')
cols = ['name', 'open_m2', 'peak_hour_arrivals', 'throughput_per_hour',
        'demand_capacity_ratio', 'overflow_residual', 'hold_upstream_min',
        'narrowest_m', 'score', 'traffic_light']
df[cols].sort_values('score', ascending=False)


In [ ]:
print('toilets required vs mapped')
print(df[['name', 'toilets_required', 'toilets_mapped', 'toilets_for_target_wait']].to_string(index=False))
print()
print('water taps required vs mapped')
print(df[['name', 'water_taps_required', 'water_points_mapped']].to_string(index=False))


### Test an assumption

The peak-day and peak-hour shares are the least certain inputs. Re-run with
different ones and see how much the conclusion actually moves.


In [ ]:
for day_share in (0.10, 0.15, 0.25):
    C.PEAK_DAY_SHARE = day_share
    r = assess.assess_site(C.SITES[0], elements=[], attendance=C.ATTENDANCE_EXPECTED)
    print(f'peak-day share {day_share:.0%} -> {r["peak_hour_arrivals"]:,} arriving at {r["name"]}')
C.PEAK_DAY_SHARE = 0.15   # restore


## 5. Build the published pages


In [ ]:
as_of = dt.date.today().isoformat()
card_paths = card.write_cards(records)
index_path = site.write_site(records, as_of)
print(f'{len(card_paths)} cards + {index_path}')


In [ ]:
# Colab blocks file:// iframes, so serve the folder and preview through the proxy.
import subprocess, time
from google.colab import output

subprocess.Popen(['python', '-m', 'http.server', '8000', '--directory', 'docs'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
output.serve_kernel_port_as_iframe(8000, height=700)
# output.serve_kernel_port_as_window(8000)   # opens in a real tab; better for the map


## 6. Publish

Download `docs/` and `outputs/`, unzip them over your local clone, commit and
push. GitHub Pages serves from the `docs/` folder of `main`.


In [ ]:
!zip -qr /content/run_outputs.zip docs outputs
from google.colab import files; files.download('/content/run_outputs.zip')


---

## What this gets wrong

- **OpenStreetMap is not an asset register.** A toilet that exists but was
  never mapped counts as missing, so sanitation gaps are an upper bound.
- **Most road widths are inferred from road class**, not measured. Each card
  says what share. This is the first thing a site visit should replace.
- **Open ground is an upper bound**: the precinct less buildings and water.
  Barriers, enclosures, vehicle routes and stalls take more away.
- **The arrival assumptions are assumptions**, reasoned from Prayagraj 2025
  rather than observed at Nashik.
- **Queueing assumes people spread evenly between blocks.** They do not, so
  real waits are worse.
- **Nothing here models behaviour.** This is a static capacity check: it
  finds sites that cannot work arithmetically. A site that passes still
  needs a dynamic simulation.

A screening tool, not a safety case. Not affiliated with or produced for any
authority.
